# CodeVerse — Teacher→Student Fine-tune on AMD Instinct (ROCm)

**Story:** the big Fireworks teacher (`glm-5p2`) generated a validated dataset of
CodeVerse theme-profile + clarifying-question completions. Here we fine-tune a small
**Qwen2.5-3B-Instruct** student with LoRA **on an AMD Instinct GPU via ROCm**, evaluate
its JSON validity, then serve it with **vLLM** as an OpenAI-compatible endpoint that the
CodeVerse app plugs into with a one-line config change.

**Before you start:** upload `codeverse_theme_sft.jsonl` (generated locally by
`amd/generate_training_data.py`) into the same folder as this notebook.

Environment: ROCm 7.2 + vLLM 0.16 + PyTorch 2.9 (the hackathon notebook image).

In [ ]:
# 1) GPU sanity check — ROCm exposes the Instinct GPU through torch.cuda
import torch
print('torch', torch.__version__)
print('gpu available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0))
print('vram GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 2) Install fine-tune deps (torch/vllm already in the image — do NOT reinstall them)
%pip install -q "transformers>=4.49" "peft>=0.14" "trl>=0.15" "datasets>=3.0" accelerate

In [ ]:
# 3) Load + inspect the distillation dataset
import json
from pathlib import Path

DATA = Path('codeverse_theme_sft.jsonl')
rows = [json.loads(line) for line in DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
print('samples:', len(rows))
print('by task:', {t: sum(1 for r in rows if r["task"] == t) for t in {r['task'] for r in rows}})

# hold out 24 samples for the post-train eval
HOLDOUT = 24
eval_rows, train_rows = rows[:HOLDOUT], rows[HOLDOUT:]
print('train:', len(train_rows), 'eval:', len(eval_rows))

In [ ]:
# 4) LoRA SFT on Qwen2.5-3B-Instruct (not gated, strong multilingual incl. Turkish)
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, device_map='auto')

def to_text(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in train_rows])

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                           target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']),
    args=SFTConfig(
        output_dir='codeverse-student-lora',
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        lr_scheduler_type='cosine',
        logging_steps=5,
        bf16=True,
        max_length=3072,
        dataset_text_field='text',
        save_strategy='epoch',
        report_to='none',
    ),
)
trainer.train()

In [ ]:
# 5) Merge LoRA into a standalone model folder (what vLLM will serve)
merged = trainer.model.merge_and_unload()
merged.save_pretrained('codeverse-student-merged')
tokenizer.save_pretrained('codeverse-student-merged')
print('merged model saved -> codeverse-student-merged')

# free training memory before vLLM grabs the GPU
del trainer, model, merged
torch.cuda.empty_cache()

In [ ]:
# 6) Serve the student with vLLM as an OpenAI-compatible API (background)
import subprocess, time, urllib.request

server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'codeverse-student-merged',
    '--served-model-name', 'codeverse-student',
    '--max-model-len', '4096',
    '--port', '8001',
], stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(60):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8001/v1/models', timeout=3)
        print('vLLM is up on :8001')
        break
    except Exception:
        pass
else:
    print('vLLM did not come up — check vllm.log')

In [ ]:
# 7) Eval: JSON validity + latency of the AMD-hosted student on held-out prompts
import json as _json, time as _time, urllib.request as _url

def extract_json_object(raw):
    text = raw.strip()
    if text.startswith('```'):
        text = text[text.index('\n') + 1:]
        if text.rstrip().endswith('```'):
            text = text.rstrip()[:-3]
    start = text.find('{')
    if start == -1:
        raise ValueError('no JSON object')
    try:
        data, _ = _json.JSONDecoder().raw_decode(text[start:])
    except _json.JSONDecodeError:
        end = text.rfind('}')
        if end <= start:
            raise ValueError('no JSON object')
        data = _json.loads(text[start:end + 1])
    if not isinstance(data, dict):
        raise ValueError('no JSON object')
    return data

def call_student(messages):
    body = _json.dumps({'model': 'codeverse-student', 'messages': messages,
                        'temperature': 0.7, 'max_tokens': 2048}).encode()
    req = _url.Request('http://localhost:8001/v1/chat/completions', data=body,
                       headers={'Content-Type': 'application/json'})
    with _url.urlopen(req, timeout=120) as resp:
        return _json.loads(resp.read())['choices'][0]['message']['content']

ok, total, latencies = 0, 0, []
for row in eval_rows:
    prompt_messages = row['messages'][:-1]  # drop the teacher's answer
    t0 = _time.time()
    try:
        raw = call_student(prompt_messages)
        data = extract_json_object(raw)
        if row['task'] == 'profile':
            assert isinstance(data.get('motifs'), list) and data['motifs']
        else:
            assert isinstance(data.get('questions'), list) and 5 <= len(data['questions']) <= 8
        ok += 1
    except Exception as exc:
        print(f'  [fail:{row["task"]}] {row["prompt"][:36]!r}: {type(exc).__name__}')
    latencies.append(_time.time() - t0)
    total += 1

print(f'\nSTUDENT on AMD: {ok}/{total} valid, avg latency {sum(latencies)/len(latencies):.1f}s')
print('(teacher baseline on Fireworks: ~100% valid, 7-8s — run the same prompts there to compare live)')

In [ ]:
# 8) Expose the endpoint publicly so the local CodeVerse app can use it
#    (cloudflared quick tunnel — free, no account needed)
import subprocess, re, time

subprocess.run(['wget', '-q', '-O', 'cloudflared',
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
subprocess.run(['chmod', '+x', 'cloudflared'])
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8001'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
    if match:
        url = match.group(0)
print('PUBLIC ENDPOINT:', url)
print()
print('Point the CodeVerse app at the AMD-hosted student — edit .env:')
print('  CODEVERSE_LLM_PROVIDER=openai_compatible')
print(f'  CODEVERSE_OPENAI_BASE_URL={url}/v1')
print('  CODEVERSE_OPENAI_API_KEY=not-needed')
print('  CODEVERSE_OPENAI_MODEL=codeverse-student')
print('then restart the backend. Theme generation now runs on AMD Instinct.')